In [1]:
import os

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv


# .env 로드
load_dotenv()


# API Key 확인
print(
    "[OK] OpenAI API 키 설정됨"
    if os.getenv("OPENAI_API_KEY")
    else "[ERROR] OpenAI API 키 미설정"
)

print(
    "[OK] LangSmith API 키 설정됨"
    if os.getenv("LANGSMITH_API_KEY")
    else "[ERROR] LangSmith API 키 미설정"
)

print(
    "[OK] Gemini API 키 설정됨"
    if os.getenv("GEMINI_API_KEY")
    else "[ERROR] Gemini API 키 미설정"
)


# LangChain 버전 확인
print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

[OK] OpenAI API 키 설정됨
[OK] LangSmith API 키 설정됨
[OK] Gemini API 키 설정됨
LangChain: 1.4.0
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


In [2]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", temperature=0, max_tokens=2048)

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "{topic}에 대해 간단히 설명해주세요."
)

chain = prompt | llm | StrOutputParser()

- invoke()

In [7]:
# 단일 입력 처리
invoke_result = chain.invoke({"topic": "지구자전"})
print(invoke_result)

지구 자전은 지구가 자신의 축을 중심으로 하루에 한 번 회전하는 운동을 말합니다. 이 자전 운동으로 인해 낮과 밤이 발생하며, 지구의 자전 주기는 약 24시간입니다. 지구의 자전 방향은 서쪽에서 동쪽으로, 즉 태양이 동쪽에서 떠서 서쪽으로 지는 방향과 같습니다. 자전은 또한 지구의 기후와 날씨 패턴에 영향을 미치며, 지구의 자전축은 약 23.5도 기울어져 있어 계절의 변화에도 중요한 역할을 합니다.


- batch()

In [11]:
# 여러 질문 처리
topics = ["지구 자전", "화산", "지진"]

batch_results = chain.batch([
    {"topic": t} for t in topics
])

for topic, result in zip(topics, batch_results):
    print(f"{topic} 설명: {result[:50]}...")


지구 자전 설명: 지구 자전은 지구가 자신의 축을 중심으로 회전하는 운동을 말합니다. 이 자전은 약 24시간...
화산 설명: 화산은 지구의 내부에서 마그마가 지표로 분출되는 현상입니다. 화산은 일반적으로 지각의 약한...
지진 설명: 지진은 지구의 지각에서 발생하는 자연 현상으로, 지각의 판이 움직이거나 충돌하면서 발생하는...


- stream()

In [ ]:
# 답변이 생성되는 대로 조금씩 출력
for chunk in chain.stream({
    "topic": "지진"
}):
    print(chunk, end="")

지진은 지구 내부의 판이 움직이면서 발생하는 자연 현상으로, 지각의 갑작스러운 변동으로 인해 발생하는 진동입니다. 이러한 진동은 지구의 내부에서 발생하는 에너지가 지각으로 방출되면서 생기며, 주로 지각판의 경계에서 발생합니다. 

지진의 강도는 리히터 규모나 모멘트 규모로 측정되며, 진앙(지진이 발생한 지점)과 진원(지하에서의 발생 지점)으로 구분됩니다. 지진은 건물과 인프라에 큰 피해를 줄 수 있으며, 해양에서 발생할 경우 쓰나미를 유발할 수도 있습니다. 

지진의 예측은 어렵지만, 특정 지역에서는 과거의 지진 데이터를 바탕으로 위험도를 평가하고 대비책을 마련하는 것이 중요합니다.

In [ ]:
# 비동기로 하나 처리
import asyncio

async def run():
    result = await chain.ainvoke({
        "topic": "해류"
    })
    print(result)

await run()


해류는 바다에서 물이 일정한 방향으로 흐르는 현상을 말합니다. 해류는 여러 가지 요인에 의해 형성되며, 주로 바람, 지구의 자전, 해수의 온도와 염도 차이, 그리고 지형적 요인 등이 영향을 미칩니다. 

해류는 크게 두 가지로 나눌 수 있습니다: 

1. **표층 해류**: 바람의 영향을 받아 수면 가까이에서 흐르는 해류로, 대양의 표면에서 주로 발생합니다. 예를 들어, 북대서양 해류와 같은 주요 해류가 여기에 해당합니다.

2. **심층 해류**: 해수의 밀도 차이에 의해 발생하는 해류로, 깊은 바다에서 흐릅니다. 이들은 지구의 기후와 생태계에 중요한 역할을 합니다.

해류는 기후 조절, 해양 생태계의 순환, 그리고 해양 생물의 이동에 큰 영향을 미치며, 항해와 어업 등에도 중요한 역할을 합니다.


### Runnable 체인 실행 방법

| 메서드 | 처리 대상 | 특징 | 예시 |
|---|---|---|---|
| `invoke()` | 입력 1개 | 하나 처리 | 질문 1개 → 답변 1개 |
| `batch()` | 입력 여러 개 | 여러 입력 처리 | 질문 10개 → 답변 10개 |
| `stream()` | 입력 1개 | 결과를 조금씩 출력 | 답변을 실시간으로 표시 |
| `ainvoke()` | 입력 1개 | 비동기로 처리 | API 호출을 기다리는 동안 다른 작업 가능 |
| `abatch()` | 입력 여러 개 | 비동기로 여러 입력 처리 | 여러 API 요청을 비동기로 처리 |
| `astream()` | 입력 1개 | 비동기 + 실시간 출력 | 비동기로 답변을 조금씩 출력 |